# 30 — Validate prioritized LIANA axes with Squidpy

**Purpose:** reproduce the historical Notebook 27 expression-support and Squidpy validation on the canonical outputs from Notebooks 25 and 29.

Historical counterpart:
- `27_squidpy_LR_validation_top_LIANA_axes_HVG8000_r0_6.ipynb`

### Canonical input policy
- Cell identity, condition, and LIANA labels come only from Notebook 25 `main_cells.tsv`.
- Candidate interactions come only from Notebook 29.
- Historical `NonMalignant_*` labels are never reconstructed.
- The historical eight thesis-facing axes are retained; the cautious `HLA_A_APLP2` axis remains outside this validation.

### Memory-safety design
- The 351,427-cell AnnData is never loaded into RAM.
- Observation and variable identifiers are read directly from HDF5.
- Only canonical main cells and the 19 single-gene ligand/receptor features needed here are streamed from the CSR `counts` layer.
- The minimal matrix is reused for expression support and the downsampled Squidpy run.
- Squidpy preprocessing exactly follows the historical notebook: `sc.pp.normalize_total(target_sum=1e4)` then `sc.pp.log1p`.

Run **top to bottom in a fresh kernel**.


In [1]:
# Imports
from pathlib import Path
from datetime import datetime
import gc
import inspect
import os
import warnings

warnings.filterwarnings("ignore")

import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import squidpy as sq

print("scanpy:", getattr(sc, "__version__", "unknown"))
print("squidpy:", getattr(sq, "__version__", "unknown"))


scanpy: 1.11.5
squidpy: 1.8.1


In [2]:
# Fixed paths and historical configuration
def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"
FIGURE_DIR = PROJECT_DIR / "figures" / "SQUIDPY_LR_validation_HVG8000_r0_6"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

OUT_PREFIX = "SQUIDPY_LR_validation_HVG8000_r0_6"
INPUT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation_malignant_states.h5ad"
MAIN_CELLS_TSV = METADATA_DIR / "LIANA_prepare_HVG8000_r0_6_main_cells.tsv"
LIANA_CANDIDATE_DETAILS = METADATA_DIR / "LIANA_final_prioritization_HVG8000_r0_6_candidate_lr_source_target_details.tsv"
LIANA_CONDITION_PRESENCE = METADATA_DIR / "LIANA_final_prioritization_HVG8000_r0_6_candidate_lr_condition_presence.tsv"
LIANA_CELLTYPE_COUNTS = METADATA_DIR / "LIANA_global_main_HVG8000_r0_6_clean_labels_celltype_counts_used.tsv"
HISTORICAL_SAMPLE_MANIFEST = METADATA_DIR / f"{OUT_PREFIX}_historical_sample_manifest.tsv"

COUNTS_LAYER = "counts"
CONDITIONS_ORDER = ["GBM", "LGG", "ndGBM", "rGBM"]
AXES_OF_INTEREST = [
    "PTN_PTPRZ1_NCL_SDC3_axis",
    "MDK_PTPRZ1_NCL_axis",
    "TIMP1_CD63_axis",
    "LGALS1_ITGB1_axis",
    "MIF_EGFR_CD74_axis",
    "VIM_CD44_axis",
    "VEGFA_angiogenesis_axis",
    "APP_CD74_axis",
]
CANONICAL_LABEL_ORDER = [
    "Malignant_AC_like_reactive_malignant",
    "Malignant_NPC_neuronal_like_malignant",
    "Malignant_OPC_NPC_like_malignant",
    "Malignant_cycling_candidate",
    "Malignant_hypoxia_stress_malignant",
    "B_Plasma",
    "Endothelial",
    "Mural_Pericyte_Fibroblast",
    "Oligodendrocyte_myelinating",
]

EXPECTED_SQUIDPY_VERSION = "1.8.1"
SQUIDPY_N_PERMS = 100
SQUIDPY_THRESHOLD = 0.10
MAX_CELLS_PER_CELLTYPE = 2500
RANDOM_SEED = 42
NORMALIZATION_TARGET_SUM = 1e4
MIN_EXPR_PROP_SUPPORT = 0.10
H5_ROW_CHUNK = 1024

EXPECTED = {
    "h5ad_cells": 351_427,
    "h5ad_genes": 15_176,
    "candidate_detail_rows": 1_298,
    "candidate_axes": 8,
    "candidate_interactions": 491,
    "candidate_lr_pairs": 15,
    "candidate_genes": 19,
    "expression_support_rows": 855,
    "candidate_expression_rows": 2_455,
    "expression_summary_rows": 40,
    "squidpy_cells": 21_167,
    "squidpy_long_rows": 1_215,
}

print("PROJECT_DIR:", PROJECT_DIR)
print("INPUT_H5AD:", INPUT_H5AD)


PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
INPUT_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_malignant_states.h5ad


In [3]:
# Helpers
def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print("[WRITE]", path, df.shape)
    return path


def savefig(path, dpi=200):
    path = Path(path)
    if path.exists():
        path.unlink()
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print("[FIG]", path)
    return path


def decode_h5_strings(values):
    return pd.Index([
        value.decode("utf-8") if isinstance(value, (bytes, np.bytes_)) else str(value)
        for value in values
    ])


def read_h5ad_axis_index(path, axis):
    with h5py.File(path, "r") as h5:
        group = h5[axis]
        key = group.attrs.get("_index", "_index")
        if isinstance(key, bytes):
            key = key.decode("utf-8")
        values = group[str(key)][:]
    return decode_h5_strings(values)


def inspect_csr_layer(path, layer):
    with h5py.File(path, "r") as h5:
        group = h5[f"layers/{layer}"]
        encoding = group.attrs.get("encoding-type", "")
        if isinstance(encoding, bytes):
            encoding = encoding.decode("utf-8")
        if encoding != "csr_matrix":
            raise RuntimeError(f"Expected CSR counts layer; found {encoding!r}.")
        shape = tuple(int(x) for x in group.attrs["shape"])
        return {
            "shape": shape,
            "data_dtype": str(group["data"].dtype),
            "indices_dtype": str(group["indices"].dtype),
            "encoding": encoding,
        }


def load_csr_rows_columns(path, layer, row_positions, column_positions, chunk_rows=1024):
    """Stream selected CSR rows and columns without materializing the full matrix."""
    row_positions = np.asarray(row_positions, dtype=np.int64)
    column_positions = np.asarray(column_positions, dtype=np.int64)
    if len(row_positions) == 0 or len(column_positions) == 0:
        raise ValueError("Selected rows and columns must be non-empty.")
    if np.any(np.diff(row_positions) <= 0):
        raise ValueError("row_positions must be sorted and unique.")
    if np.any(np.diff(column_positions) <= 0):
        raise ValueError("column_positions must be sorted and unique.")

    with h5py.File(path, "r") as h5:
        group = h5[f"layers/{layer}"]
        shape = tuple(int(x) for x in group.attrs["shape"])
        source_indptr = np.asarray(group["indptr"][:], dtype=np.int64)
        selected_rows = np.zeros(shape[0], dtype=bool)
        selected_rows[row_positions] = True
        column_map = np.full(shape[1], -1, dtype=np.int32)
        column_map[column_positions] = np.arange(len(column_positions), dtype=np.int32)

        data_parts = []
        index_parts = []
        row_nnz = []
        for start in range(0, shape[0], chunk_rows):
            stop = min(start + chunk_rows, shape[0])
            relative_rows = np.flatnonzero(selected_rows[start:stop])
            if len(relative_rows) == 0:
                continue
            pointer_start = int(source_indptr[start])
            pointer_stop = int(source_indptr[stop])
            chunk_data = np.asarray(group["data"][pointer_start:pointer_stop])
            chunk_indices = np.asarray(group["indices"][pointer_start:pointer_stop])
            chunk_indptr = source_indptr[start:stop + 1] - pointer_start
            for relative_row in relative_rows:
                p0 = int(chunk_indptr[relative_row])
                p1 = int(chunk_indptr[relative_row + 1])
                mapped = column_map[chunk_indices[p0:p1]]
                keep = mapped >= 0
                kept_data = chunk_data[p0:p1][keep]
                kept_indices = mapped[keep]
                data_parts.append(kept_data)
                index_parts.append(kept_indices)
                row_nnz.append(int(keep.sum()))

        data = np.concatenate(data_parts) if data_parts else np.array([], dtype=np.float32)
        indices = np.concatenate(index_parts).astype(np.int32, copy=False) if index_parts else np.array([], dtype=np.int32)
        indptr = np.empty(len(row_nnz) + 1, dtype=np.int64)
        indptr[0] = 0
        np.cumsum(np.asarray(row_nnz, dtype=np.int64), out=indptr[1:])

    matrix = sp.csr_matrix(
        (data, indices, indptr),
        shape=(len(row_positions), len(column_positions)),
        copy=False,
    )
    matrix.sort_indices()
    return matrix


def matrix_mean_pct(X, cell_indices):
    sub = X[cell_indices, :]
    pct = np.asarray((sub > 0).mean(axis=0)).ravel()
    mean = np.asarray(sub.mean(axis=0)).ravel()
    return pct, mean


def safe_neglog10(values):
    values = pd.to_numeric(values, errors="coerce")
    return -np.log10(np.maximum(values, 1e-300))


def table_to_long(table, value_column):
    rows = []
    if table is None or table.empty:
        return pd.DataFrame(columns=["ligand", "receptor", "lr_pair", "source_clean", "target_clean", value_column])
    for interaction, row in table.iterrows():
        ligand, receptor = map(str, interaction[:2])
        for source_target, value in row.items():
            source, target = map(str, source_target[:2])
            rows.append({
                "ligand": ligand,
                "receptor": receptor,
                "lr_pair": f"{ligand}→{receptor}",
                "source_clean": source,
                "target_clean": target,
                value_column: value,
            })
    return pd.DataFrame(rows)


In [4]:
# Read and audit canonical inputs without loading the full AnnData
required_paths = [
    INPUT_H5AD,
    MAIN_CELLS_TSV,
    LIANA_CANDIDATE_DETAILS,
    LIANA_CONDITION_PRESENCE,
    LIANA_CELLTYPE_COUNTS,
    HISTORICAL_SAMPLE_MANIFEST,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing_paths))
if str(getattr(sq, "__version__", "unknown")) != EXPECTED_SQUIDPY_VERSION:
    raise RuntimeError(
        f"Historical validation requires Squidpy {EXPECTED_SQUIDPY_VERSION}; "
        f"found {getattr(sq, '__version__', 'unknown')}."
    )

obs_names = read_h5ad_axis_index(INPUT_H5AD, "obs")
var_names = read_h5ad_axis_index(INPUT_H5AD, "var")
layer_info = inspect_csr_layer(INPUT_H5AD, COUNTS_LAYER)
if layer_info["shape"] != (EXPECTED["h5ad_cells"], EXPECTED["h5ad_genes"]):
    raise AssertionError(f"Unexpected counts shape: {layer_info['shape']}")
if len(obs_names) != len(obs_names.unique()) or len(var_names) != len(var_names.unique()):
    raise AssertionError("H5AD observation or variable identifiers are not unique.")

main_cells = pd.read_csv(
    MAIN_CELLS_TSV,
    sep="\t",
    usecols=["cell_id", "condition", "liana_celltype_main"],
)
candidates_raw = pd.read_csv(LIANA_CANDIDATE_DETAILS, sep="\t")
condition_presence = pd.read_csv(LIANA_CONDITION_PRESENCE, sep="\t")
celltype_counts = pd.read_csv(LIANA_CELLTYPE_COUNTS, sep="\t")
historical_sample_manifest = pd.read_csv(HISTORICAL_SAMPLE_MANIFEST, sep="\t")

if main_cells["cell_id"].isna().any() or main_cells["cell_id"].duplicated().any():
    raise AssertionError("Notebook 25 main-cell identifiers contain NA or duplicates.")
if set(main_cells["condition"].astype(str)) != set(CONDITIONS_ORDER):
    raise AssertionError("Canonical main-cell conditions differ from the four expected conditions.")
if set(main_cells["liana_celltype_main"].astype(str)) != set(CANONICAL_LABEL_ORDER):
    raise AssertionError("Canonical Notebook 25 LIANA labels differ from the expected nine labels.")
if len(candidates_raw) != EXPECTED["candidate_detail_rows"]:
    raise AssertionError("Notebook 29 candidate-detail row count drifted.")

for column in ["source_clean", "target_clean"]:
    values = candidates_raw[column].astype(str)
    if values.str.startswith("NonMalignant_").any() or values.str.contains("Malignant_non_malignant", regex=False).any():
        raise AssertionError(f"Historical or malformed labels found in {column}.")

readiness = pd.DataFrame([
    {"item": "input_h5ad", "value": str(INPUT_H5AD), "status": "found"},
    {"item": "h5ad_n_cells", "value": len(obs_names), "status": "ok"},
    {"item": "h5ad_n_genes", "value": len(var_names), "status": "ok"},
    {"item": "canonical_main_cells", "value": len(main_cells), "status": "ok"},
    {"item": "canonical_celltypes", "value": main_cells["liana_celltype_main"].nunique(), "status": "ok"},
    {"item": "candidate_detail_rows", "value": len(candidates_raw), "status": "ok"},
    {"item": "historical_sample_manifest_rows", "value": len(historical_sample_manifest), "status": "ok"},
    {"item": "squidpy_version", "value": sq.__version__, "status": "exact_historical"},
    {"item": "counts_encoding", "value": layer_info["encoding"], "status": "memory_safe_supported"},
])
write_tsv_replace(readiness, METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv")
display(readiness)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_readiness.tsv (9, 3)


,item,value,status
0,input_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,found
1,h5ad_n_cells,351427,ok
2,h5ad_n_genes,15176,ok
3,canonical_main_cells,184066,ok
4,canonical_celltypes,9,ok
5,candidate_detail_rows,1298,ok
6,historical_sample_manifest_rows,21167,ok
7,squidpy_version,1.8.1,exact_historical
8,counts_encoding,csr_matrix,memory_safe_supported


In [5]:
# Build the fixed historical eight-axis candidate set with canonical labels
required_columns = {"candidate_axis", "ligand", "receptor", "source_clean", "target_clean"}
missing_columns = required_columns - set(candidates_raw.columns)
if missing_columns:
    raise KeyError(f"Notebook 29 candidate details are missing: {sorted(missing_columns)}")

candidates = candidates_raw.loc[
    candidates_raw["candidate_axis"].isin(AXES_OF_INTEREST)
].copy()
candidates["ligand"] = candidates["ligand"].astype(str)
candidates["receptor"] = candidates["receptor"].astype(str)
candidates["source_clean"] = candidates["source_clean"].astype(str)
candidates["target_clean"] = candidates["target_clean"].astype(str)
candidates["lr_pair"] = candidates["ligand"] + "→" + candidates["receptor"]
candidates = candidates.drop_duplicates(
    ["candidate_axis", "source_clean", "target_clean", "ligand", "receptor"]
).reset_index(drop=True)

var_set = set(var_names)
candidates["ligand_in_var"] = candidates["ligand"].isin(var_set)
candidates["receptor_in_var"] = candidates["receptor"].isin(var_set)
candidates["single_gene_pair_available"] = candidates["ligand_in_var"] & candidates["receptor_in_var"]

available = candidates.loc[candidates["single_gene_pair_available"]].copy()
interactions = sorted(set(zip(available["ligand"], available["receptor"])))
genes = sorted({gene for pair in interactions for gene in pair})

candidate_checkpoints = {
    "candidate_axes": candidates["candidate_axis"].nunique(),
    "candidate_interactions": len(candidates),
    "candidate_lr_pairs": len(interactions),
    "candidate_genes": len(genes),
}
for name, observed in candidate_checkpoints.items():
    if observed != EXPECTED[name]:
        raise AssertionError(f"{name}: expected {EXPECTED[name]}, observed {observed}")

inventory = (
    candidates.groupby("candidate_axis", observed=True)
    .agg(
        n_rows=("lr_pair", "size"),
        n_unique_lr=("lr_pair", "nunique"),
        n_source_target=("source_clean", lambda x: len(set(zip(candidates.loc[x.index, "source_clean"], candidates.loc[x.index, "target_clean"])))),
        n_single_gene_available=("single_gene_pair_available", "sum"),
    )
    .reindex(AXES_OF_INTEREST)
    .reset_index()
)
write_tsv_replace(candidates, METADATA_DIR / f"{OUT_PREFIX}_candidate_interactions.tsv")
write_tsv_replace(inventory, METADATA_DIR / f"{OUT_PREFIX}_candidate_axis_inventory.tsv")
display(inventory)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_candidate_interactions.tsv (491, 21)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_candidate_axis_inventory.tsv (8, 5)


,candidate_axis,n_rows,n_unique_lr,n_source_target,n_single_gene_available
0,PTN_PTPRZ1_NCL_SDC3_axis,162,4,62,162
1,MDK_PTPRZ1_NCL_axis,110,4,54,110
2,TIMP1_CD63_axis,64,1,64,64
3,LGALS1_ITGB1_axis,46,1,46,46
4,MIF_EGFR_CD74_axis,45,1,45,45
5,VIM_CD44_axis,31,1,31,31
6,VEGFA_angiogenesis_axis,16,2,14,16
7,APP_CD74_axis,17,1,17,17


In [6]:
# Map canonical cells to H5AD rows and stream only the 19 required genes
positions = obs_names.get_indexer(main_cells["cell_id"].astype(str))
if (positions < 0).any() or len(np.unique(positions)) != len(positions):
    raise AssertionError("Canonical main cells do not map one-to-one to H5AD rows.")

main_cells = main_cells.assign(h5ad_position=positions).sort_values("h5ad_position").reset_index(drop=True)
row_positions = main_cells["h5ad_position"].to_numpy(dtype=np.int64)
gene_positions_original = np.array([var_names.get_loc(gene) for gene in genes], dtype=np.int64)
gene_order = np.argsort(gene_positions_original)
column_positions = gene_positions_original[gene_order]
streamed_genes = [genes[i] for i in gene_order]

X_selected = load_csr_rows_columns(
    INPUT_H5AD,
    COUNTS_LAYER,
    row_positions,
    column_positions,
    chunk_rows=H5_ROW_CHUNK,
)
if X_selected.shape != (len(main_cells), len(genes)):
    raise AssertionError(f"Streamed matrix has unexpected shape {X_selected.shape}.")
if np.any(~np.isfinite(X_selected.data)) or np.any(X_selected.data < 0):
    raise AssertionError("Streamed counts contain invalid values.")

# Restore alphabetical gene order used by the historical notebook.
restore_order = np.array([streamed_genes.index(gene) for gene in genes], dtype=np.int64)
X_selected = X_selected[:, restore_order].tocsr()

counts_observed = (
    main_cells["liana_celltype_main"].value_counts()
    .reindex(CANONICAL_LABEL_ORDER)
    .rename_axis("squidpy_liana_celltype")
    .reset_index(name="n_cells")
)
expected_count_map = dict(zip(celltype_counts["liana_celltype"].astype(str), pd.to_numeric(celltype_counts["observed_n_cells"])))
counts_observed["expected_n_cells"] = counts_observed["squidpy_liana_celltype"].map(expected_count_map).astype(int)
counts_observed["match"] = counts_observed["n_cells"] == counts_observed["expected_n_cells"]
if not counts_observed["match"].all():
    raise AssertionError("Canonical cell-type counts differ from Notebook 27 inputs.")
write_tsv_replace(counts_observed, METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_used.tsv")

memory_audit = pd.DataFrame([
    {"item": "full_anndata_loaded", "value": False},
    {"item": "streamed_cells", "value": len(main_cells)},
    {"item": "streamed_genes", "value": len(genes)},
    {"item": "streamed_nnz", "value": int(X_selected.nnz)},
    {"item": "streamed_matrix_mb", "value": round((X_selected.data.nbytes + X_selected.indices.nbytes + X_selected.indptr.nbytes) / 1024**2, 3)},
    {"item": "source_counts_dtype", "value": layer_info["data_dtype"]},
])
write_tsv_replace(memory_audit, METADATA_DIR / f"{OUT_PREFIX}_memory_audit.tsv")
display(counts_observed)
display(memory_audit)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_celltype_counts_used.tsv (9, 4)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_memory_audit.tsv (6, 2)


,squidpy_liana_celltype,n_cells,expected_n_cells,match
0,Malignant_AC_like_reactive_malignant,22525,22525,True
1,Malignant_NPC_neuronal_like_malignant,6183,6183,True
2,Malignant_OPC_NPC_like_malignant,61646,61646,True
3,Malignant_cycling_candidate,38634,38634,True
4,Malignant_hypoxia_stress_malignant,32896,32896,True
5,B_Plasma,1167,1167,True
6,Endothelial,2640,2640,True
7,Mural_Pericyte_Fibroblast,3566,3566,True
8,Oligodendrocyte_myelinating,14809,14809,True


,item,value
0,full_anndata_loaded,False
1,streamed_cells,184066
2,streamed_genes,19
3,streamed_nnz,1952720
4,streamed_matrix_mb,15.6
5,source_counts_dtype,int32


In [7]:
# Exact expression-support validation across global and condition-specific groups
celltypes = main_cells["liana_celltype_main"].astype(str).to_numpy()
conditions = main_cells["condition"].astype(str).to_numpy()
support_rows = []

for condition in ["GLOBAL", *CONDITIONS_ORDER]:
    condition_mask = np.ones(len(main_cells), dtype=bool) if condition == "GLOBAL" else conditions == condition
    for celltype in CANONICAL_LABEL_ORDER:
        indices = np.flatnonzero(condition_mask & (celltypes == celltype))
        if len(indices) == 0:
            continue
        pct, mean = matrix_mean_pct(X_selected, indices)
        for gene, pct_value, mean_value in zip(genes, pct, mean):
            support_rows.append({
                "condition": condition,
                "celltype": celltype,
                "gene": gene,
                "n_cells": len(indices),
                "pct_expr": float(pct_value),
                "mean_count": float(mean_value),
            })

expression_support = pd.DataFrame(support_rows)
if len(expression_support) != EXPECTED["expression_support_rows"]:
    raise AssertionError(f"Expression-support rows drifted: {len(expression_support)}")
write_tsv_replace(expression_support, METADATA_DIR / f"{OUT_PREFIX}_gene_expression_support_by_celltype_condition.tsv")

support_index = expression_support.set_index(["condition", "celltype", "gene"])
candidate_rows = []
for row in available.itertuples(index=False):
    for condition in ["GLOBAL", *CONDITIONS_ORDER]:
        ligand_support = support_index.loc[(condition, row.source_clean, row.ligand)]
        receptor_support = support_index.loc[(condition, row.target_clean, row.receptor)]
        ligand_pct = float(ligand_support["pct_expr"])
        receptor_pct = float(receptor_support["pct_expr"])
        candidate_rows.append({
            "condition": condition,
            "candidate_axis": row.candidate_axis,
            "source_clean": row.source_clean,
            "target_clean": row.target_clean,
            "ligand": row.ligand,
            "receptor": row.receptor,
            "lr_pair": row.lr_pair,
            "source_n_cells": int(ligand_support["n_cells"]),
            "target_n_cells": int(receptor_support["n_cells"]),
            "ligand_source_pct_expr": ligand_pct,
            "receptor_target_pct_expr": receptor_pct,
            "ligand_source_mean_count": float(ligand_support["mean_count"]),
            "receptor_target_mean_count": float(receptor_support["mean_count"]),
            "expression_support_pass_10pct": ligand_pct >= MIN_EXPR_PROP_SUPPORT and receptor_pct >= MIN_EXPR_PROP_SUPPORT,
            "min_ligand_receptor_pct": min(ligand_pct, receptor_pct),
        })

candidate_expression = pd.DataFrame(candidate_rows)
global_expression = candidate_expression.loc[candidate_expression["condition"] == "GLOBAL"].copy()
expression_summary = (
    candidate_expression.groupby(["condition", "candidate_axis"], observed=True)
    .agg(
        n_candidate_interactions=("lr_pair", "size"),
        n_pass_10pct=("expression_support_pass_10pct", "sum"),
        frac_pass_10pct=("expression_support_pass_10pct", "mean"),
        median_min_pct=("min_ligand_receptor_pct", "median"),
        median_ligand_source_pct=("ligand_source_pct_expr", "median"),
        median_receptor_target_pct=("receptor_target_pct_expr", "median"),
    )
    .reset_index()
)
if len(candidate_expression) != EXPECTED["candidate_expression_rows"] or len(expression_summary) != EXPECTED["expression_summary_rows"]:
    raise AssertionError("Historical expression-support output dimensions drifted.")

write_tsv_replace(candidate_expression, METADATA_DIR / f"{OUT_PREFIX}_candidate_expression_support_by_condition.tsv")
write_tsv_replace(global_expression, METADATA_DIR / f"{OUT_PREFIX}_candidate_expression_support_global.tsv")
write_tsv_replace(expression_summary, METADATA_DIR / f"{OUT_PREFIX}_candidate_expression_support_summary.tsv")
display(expression_summary)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_gene_expression_support_by_celltype_condition.tsv (855, 6)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_candidate_expression_support_by_condition.tsv (2455, 15)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_candidate_expression_support_global.tsv (491, 15)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_candidate_expression_support_summary.tsv (40, 8)


,condition,candidate_axis,n_candidate_interactions,n_pass_10pct,frac_pass_10pct,median_min_pct,median_ligand_source_pct,median_receptor_target_pct
0,GBM,APP_CD74_axis,17,17,1.000000,0.218324,0.667641,0.218324
1,GBM,LGALS1_ITGB1_axis,46,46,1.000000,0.361600,0.727700,0.361600
2,GBM,MDK_PTPRZ1_NCL_axis,110,110,1.000000,0.364864,0.552340,0.544201
3,GBM,MIF_EGFR_CD74_axis,45,45,1.000000,0.436605,0.680498,0.471709
4,GBM,PTN_PTPRZ1_NCL_SDC3_axis,162,159,0.981481,0.367576,0.802371,0.451956
5,GBM,TIMP1_CD63_axis,64,64,1.000000,0.536772,0.536772,0.861106
6,GBM,VEGFA_angiogenesis_axis,16,16,1.000000,0.243392,0.754774,0.516443
7,GBM,VIM_CD44_axis,31,25,0.806452,0.165395,0.905297,0.165395
8,GLOBAL,APP_CD74_axis,17,17,1.000000,0.478788,0.766482,0.478788
9,GLOBAL,LGALS1_ITGB1_axis,46,46,1.000000,0.469172,0.803127,0.469172


In [8]:
# Reproduce the exact historical downsample through the audited stable-ID manifest.
required_manifest_columns = {
    "historical_sample_order", "current_cell_id", "current_h5ad_position",
    "current_canonical_label", "label_match",
}
missing_manifest_columns = required_manifest_columns - set(historical_sample_manifest.columns)
if missing_manifest_columns:
    raise KeyError(f"Historical sample manifest is missing: {sorted(missing_manifest_columns)}")

sample_manifest = historical_sample_manifest.sort_values("historical_sample_order").reset_index(drop=True)
if len(sample_manifest) != EXPECTED["squidpy_cells"]:
    raise AssertionError(f"Historical sample manifest has {len(sample_manifest)} rows.")
if sample_manifest["historical_sample_order"].duplicated().any() or sample_manifest["current_cell_id"].duplicated().any():
    raise AssertionError("Historical sample manifest contains duplicate order or cell identifiers.")
if not sample_manifest["label_match"].astype(str).str.lower().eq("true").all():
    raise AssertionError("Historical sample manifest contains a failed label mapping.")

minimal_position_by_id = pd.Series(
    np.arange(len(main_cells), dtype=np.int64),
    index=main_cells["cell_id"].astype(str),
)
selected = sample_manifest["current_cell_id"].astype(str).map(minimal_position_by_id)
if selected.isna().any():
    raise AssertionError("One or more historical sample cells are absent from the streamed canonical matrix.")
selected = selected.to_numpy(dtype=np.int64)

observed_sample_labels = celltypes[selected]
expected_sample_labels = sample_manifest["current_canonical_label"].astype(str).to_numpy()
if not np.array_equal(observed_sample_labels, expected_sample_labels):
    raise AssertionError("Manifest labels differ from canonical Notebook 25 labels after cell mapping.")
if not np.array_equal(
    main_cells.loc[selected, "h5ad_position"].to_numpy(dtype=np.int64),
    pd.to_numeric(sample_manifest["current_h5ad_position"]).to_numpy(dtype=np.int64),
):
    raise AssertionError("Manifest H5AD positions differ from the current canonical H5AD.")

obs_minimal = pd.DataFrame(
    {
        "squidpy_liana_celltype": pd.Categorical(
            celltypes[selected], categories=CANONICAL_LABEL_ORDER, ordered=True
        )
    },
    index=main_cells.loc[selected, "cell_id"].astype(str),
)
adata_sq = ad.AnnData(
    X=X_selected[selected, :].copy(),
    obs=obs_minimal,
    var=pd.DataFrame(index=pd.Index(genes, name="gene")),
)
sc.pp.normalize_total(adata_sq, target_sum=NORMALIZATION_TARGET_SUM)
sc.pp.log1p(adata_sq)
if np.any(~np.isfinite(adata_sq.X.data)):
    raise AssertionError("Non-finite values after historical Scanpy preprocessing.")

kwargs = dict(
    adata=adata_sq,
    cluster_key="squidpy_liana_celltype",
    interactions=interactions,
    threshold=SQUIDPY_THRESHOLD,
    copy=True,
    use_raw=False,
    n_perms=SQUIDPY_N_PERMS,
    seed=RANDOM_SEED,
)
signature = inspect.signature(sq.gr.ligrec)
if "corr_method" in signature.parameters:
    kwargs["corr_method"] = "fdr_bh"
elif "fdr_method" in signature.parameters:
    kwargs["fdr_method"] = "fdr_bh"
if "corr_axis" in signature.parameters:
    kwargs["corr_axis"] = "interactions"
elif "fdr_axis" in signature.parameters:
    kwargs["fdr_axis"] = "interactions"
if "numba_parallel" in signature.parameters:
    kwargs["numba_parallel"] = False

print("[RUN Squidpy]", adata_sq, "interactions=", len(interactions))
result = sq.gr.ligrec(**kwargs)
if not isinstance(result, dict) or "means" not in result or "pvalues" not in result:
    raise RuntimeError("Unexpected Squidpy ligrec result structure.")

squidpy_means = table_to_long(result["means"], "squidpy_mean")
squidpy_pvalues = table_to_long(result["pvalues"], "squidpy_pvalue")
if len(squidpy_means) != EXPECTED["squidpy_long_rows"] or len(squidpy_pvalues) != EXPECTED["squidpy_long_rows"]:
    raise AssertionError("Squidpy long-table dimensions drifted from the historical run.")

run_status = pd.DataFrame([
    {"item": "squidpy_status", "value": "available"},
    {"item": "squidpy_version", "value": sq.__version__},
    {"item": "squidpy_run_status", "value": "completed"},
    {"item": "squidpy_n_cells", "value": adata_sq.n_obs},
    {"item": "squidpy_n_genes", "value": adata_sq.n_vars},
    {"item": "squidpy_n_candidate_interactions", "value": len(interactions)},
    {"item": "squidpy_n_perms", "value": SQUIDPY_N_PERMS},
    {"item": "squidpy_threshold", "value": SQUIDPY_THRESHOLD},
    {"item": "random_seed", "value": RANDOM_SEED},
    {"item": "sample_selection", "value": "audited_historical_stable_id_manifest"},
    {"item": "normalization", "value": "scanpy_normalize_total_1e4_then_log1p"},
])
write_tsv_replace(run_status, METADATA_DIR / f"{OUT_PREFIX}_squidpy_run_status.tsv")
write_tsv_replace(squidpy_means, METADATA_DIR / f"{OUT_PREFIX}_squidpy_means_long.tsv")
write_tsv_replace(squidpy_pvalues, METADATA_DIR / f"{OUT_PREFIX}_squidpy_pvalues_long.tsv")
display(run_status)


[RUN Squidpy] AnnData object with n_obs × n_vars = 21167 × 19
    obs: 'squidpy_liana_celltype'
    uns: 'log1p' interactions= 15


100%|██████████| 100/100 [00:08<00:00, 11.57permutation/s]


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_squidpy_run_status.tsv (11, 2)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_squidpy_means_long.tsv (1215, 6)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_squidpy_pvalues_long.tsv (1215, 6)


,item,value
0,squidpy_status,available
1,squidpy_version,1.8.1
2,squidpy_run_status,completed
3,squidpy_n_cells,21167
4,squidpy_n_genes,19
5,squidpy_n_candidate_interactions,15
6,squidpy_n_perms,100
7,squidpy_threshold,0.1
8,random_seed,42
9,sample_selection,audited_historical_stable_id_manifest


In [9]:
# Candidate overlap and thesis-facing validation summary
merge_columns = ["source_clean", "target_clean", "ligand", "receptor", "lr_pair"]
squidpy_values = squidpy_pvalues.merge(squidpy_means, on=merge_columns, how="left", validate="one_to_one")
overlap = candidates.merge(squidpy_values, on=merge_columns, how="left", validate="many_to_one")
overlap["squidpy_tested"] = overlap["squidpy_pvalue"].notna()
overlap["squidpy_pvalue_le_0_05"] = pd.to_numeric(overlap["squidpy_pvalue"], errors="coerce") <= 0.05
overlap["squidpy_neglog10_p"] = safe_neglog10(overlap["squidpy_pvalue"])
if len(overlap) != EXPECTED["candidate_interactions"] or not overlap["squidpy_tested"].all():
    raise AssertionError("Not every canonical candidate source-target row mapped to Squidpy output.")

overlap_summary = (
    overlap.groupby("candidate_axis", observed=True)
    .agg(
        n_candidate_interactions=("lr_pair", "size"),
        n_squidpy_tested=("squidpy_tested", "sum"),
        n_squidpy_p_le_0_05=("squidpy_pvalue_le_0_05", "sum"),
        frac_squidpy_p_le_0_05=("squidpy_pvalue_le_0_05", "mean"),
        min_squidpy_pvalue=("squidpy_pvalue", "min"),
        median_squidpy_mean=("squidpy_mean", "median"),
    )
    .reset_index()
)

global_summary = expression_summary.loc[expression_summary["condition"] == "GLOBAL"].copy()
summary = global_summary.merge(overlap_summary, on="candidate_axis", how="left", validate="one_to_one")

def interpret_row(row):
    expression_ok = row["frac_pass_10pct"] >= 0.5
    squidpy_ok = row["n_squidpy_p_le_0_05"] > 0
    if expression_ok and squidpy_ok:
        return "Supported by expression QC and at least one Squidpy ligrec signal"
    if expression_ok:
        return "Expression-supported; no significant Squidpy support under this lightweight setting"
    if squidpy_ok:
        return "Partial expression support with at least one Squidpy signal; inspect manually"
    return "Weak or partial support under the fixed validation settings"

summary["validation_interpretation"] = summary.apply(interpret_row, axis=1)
summary["suggested_reporting"] = summary["candidate_axis"].map({
    "PTN_PTPRZ1_NCL_SDC3_axis": "Prioritized LIANA axis; report as recurrent candidate malignant/progenitor-like signaling.",
    "MDK_PTPRZ1_NCL_axis": "Prioritized LIANA axis; report as candidate growth-factor-like communication.",
    "TIMP1_CD63_axis": "Candidate ECM, remodeling, and stress-associated axis.",
    "LGALS1_ITGB1_axis": "Candidate adhesion and ECM-associated axis.",
    "MIF_EGFR_CD74_axis": "Candidate inflammatory and growth-signaling axis.",
    "VIM_CD44_axis": "Candidate mesenchymal and stress-associated axis.",
    "VEGFA_angiogenesis_axis": "Candidate angiogenic axis with condition-specific interpretation.",
    "APP_CD74_axis": "Candidate immune and inflammatory axis; interpret cautiously.",
})

write_tsv_replace(overlap, METADATA_DIR / f"{OUT_PREFIX}_squidpy_candidate_overlap_details.tsv")
write_tsv_replace(overlap_summary, METADATA_DIR / f"{OUT_PREFIX}_squidpy_candidate_overlap_summary.tsv")
write_tsv_replace(summary, METADATA_DIR / f"{OUT_PREFIX}_thesis_validation_summary.tsv")
display(overlap_summary)
display(summary)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_squidpy_candidate_overlap_details.tsv (491, 26)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_squidpy_candidate_overlap_summary.tsv (8, 7)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_thesis_validation_summary.tsv (8, 16)


,candidate_axis,n_candidate_interactions,n_squidpy_tested,n_squidpy_p_le_0_05,frac_squidpy_p_le_0_05,min_squidpy_pvalue,median_squidpy_mean
0,APP_CD74_axis,17,17,15,0.882353,0.0,3.899519
1,LGALS1_ITGB1_axis,46,46,29,0.630435,0.0,4.152911
2,MDK_PTPRZ1_NCL_axis,110,110,91,0.827273,0.0,3.716051
3,MIF_EGFR_CD74_axis,45,45,30,0.666667,0.0,3.916696
4,PTN_PTPRZ1_NCL_SDC3_axis,162,162,103,0.635802,0.0,4.531198
5,TIMP1_CD63_axis,64,64,38,0.593750,0.0,5.157908
6,VEGFA_angiogenesis_axis,16,16,16,1.000000,0.0,3.545603
7,VIM_CD44_axis,31,31,27,0.870968,0.0,4.298276


,condition,candidate_axis,n_candidate_interactions_x,n_pass_10pct,frac_pass_10pct,median_min_pct,median_ligand_source_pct,median_receptor_target_pct,n_candidate_interactions_y,n_squidpy_tested,n_squidpy_p_le_0_05,frac_squidpy_p_le_0_05,min_squidpy_pvalue,median_squidpy_mean,validation_interpretation,suggested_reporting
0,GLOBAL,APP_CD74_axis,17,17,1.0,0.478788,0.766482,0.478788,17,17,15,0.882353,0.0,3.899519,Supported by expression QC and at least one Sq...,Candidate immune and inflammatory axis; interp...
1,GLOBAL,LGALS1_ITGB1_axis,46,46,1.0,0.469172,0.803127,0.469172,46,46,29,0.630435,0.0,4.152911,Supported by expression QC and at least one Sq...,Candidate adhesion and ECM-associated axis.
2,GLOBAL,MDK_PTPRZ1_NCL_axis,110,110,1.0,0.516752,0.581756,0.657588,110,110,91,0.827273,0.0,3.716051,Supported by expression QC and at least one Sq...,Prioritized LIANA axis; report as candidate gr...
3,GLOBAL,MIF_EGFR_CD74_axis,45,45,1.0,0.491188,0.818372,0.491188,45,45,30,0.666667,0.0,3.916696,Supported by expression QC and at least one Sq...,Candidate inflammatory and growth-signaling axis.
4,GLOBAL,PTN_PTPRZ1_NCL_SDC3_axis,162,162,1.0,0.598864,0.828099,0.612324,162,162,103,0.635802,0.0,4.531198,Supported by expression QC and at least one Sq...,Prioritized LIANA axis; report as recurrent ca...
5,GLOBAL,TIMP1_CD63_axis,64,64,1.0,0.677468,0.685660,0.902653,64,64,38,0.593750,0.0,5.157908,Supported by expression QC and at least one Sq...,"Candidate ECM, remodeling, and stress-associat..."
6,GLOBAL,VEGFA_angiogenesis_axis,16,16,1.0,0.395621,0.813260,0.645776,16,16,16,1.000000,0.0,3.545603,Supported by expression QC and at least one Sq...,Candidate angiogenic axis with condition-speci...
7,GLOBAL,VIM_CD44_axis,31,31,1.0,0.489063,0.941543,0.489063,31,31,27,0.870968,0.0,4.298276,Supported by expression QC and at least one Sq...,Candidate mesenchymal and stress-associated axis.


In [10]:
# Historical structural checkpoints and canonical-label audit
checkpoint_rows = [
    {"checkpoint": "h5ad_cells", "expected_historical": EXPECTED["h5ad_cells"], "observed": len(obs_names)},
    {"checkpoint": "h5ad_genes", "expected_historical": EXPECTED["h5ad_genes"], "observed": len(var_names)},
    {"checkpoint": "candidate_detail_rows", "expected_historical": EXPECTED["candidate_detail_rows"], "observed": len(candidates_raw)},
    {"checkpoint": "candidate_axes", "expected_historical": EXPECTED["candidate_axes"], "observed": candidates["candidate_axis"].nunique()},
    {"checkpoint": "candidate_interactions", "expected_historical": EXPECTED["candidate_interactions"], "observed": len(candidates)},
    {"checkpoint": "candidate_lr_pairs", "expected_historical": EXPECTED["candidate_lr_pairs"], "observed": len(interactions)},
    {"checkpoint": "candidate_genes", "expected_historical": EXPECTED["candidate_genes"], "observed": len(genes)},
    {"checkpoint": "expression_support_rows", "expected_historical": EXPECTED["expression_support_rows"], "observed": len(expression_support)},
    {"checkpoint": "candidate_expression_rows", "expected_historical": EXPECTED["candidate_expression_rows"], "observed": len(candidate_expression)},
    {"checkpoint": "expression_summary_rows", "expected_historical": EXPECTED["expression_summary_rows"], "observed": len(expression_summary)},
    {"checkpoint": "squidpy_cells", "expected_historical": EXPECTED["squidpy_cells"], "observed": adata_sq.n_obs},
    {"checkpoint": "squidpy_means_rows", "expected_historical": EXPECTED["squidpy_long_rows"], "observed": len(squidpy_means)},
    {"checkpoint": "squidpy_pvalues_rows", "expected_historical": EXPECTED["squidpy_long_rows"], "observed": len(squidpy_pvalues)},
]
historical_audit = pd.DataFrame(checkpoint_rows)
historical_audit["match"] = historical_audit["observed"] == historical_audit["expected_historical"]
if not historical_audit["match"].all():
    raise AssertionError("One or more historical structural checkpoints failed.")

output_labels = set(candidates["source_clean"]) | set(candidates["target_clean"]) | set(squidpy_means["source_clean"]) | set(squidpy_means["target_clean"])
label_audit = pd.DataFrame([
    {"check": "all_output_labels_are_canonical", "pass": output_labels <= set(CANONICAL_LABEL_ORDER)},
    {"check": "no_historical_nonmalignant_labels", "pass": not any(label.startswith("NonMalignant_") for label in output_labels)},
    {"check": "no_malformed_malignant_nonmalignant_labels", "pass": not any("Malignant_non_malignant" in label for label in output_labels)},
    {"check": "notebook25_labels_used_directly", "pass": True},
    {"check": "historical_aliases_used_for_computation", "pass": False},
])
if not label_audit.loc[label_audit["check"] != "historical_aliases_used_for_computation", "pass"].all():
    raise AssertionError("Canonical label audit failed.")

write_tsv_replace(historical_audit, METADATA_DIR / f"{OUT_PREFIX}_historical_checkpoint_audit.tsv")
write_tsv_replace(label_audit, METADATA_DIR / f"{OUT_PREFIX}_canonical_label_audit.tsv")
display(historical_audit)
display(label_audit)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_historical_checkpoint_audit.tsv (13, 4)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_canonical_label_audit.tsv (5, 2)


,checkpoint,expected_historical,observed,match
0,h5ad_cells,351427,351427,True
1,h5ad_genes,15176,15176,True
2,candidate_detail_rows,1298,1298,True
3,candidate_axes,8,8,True
4,candidate_interactions,491,491,True
5,candidate_lr_pairs,15,15,True
6,candidate_genes,19,19,True
7,expression_support_rows,855,855,True
8,candidate_expression_rows,2455,2455,True
9,expression_summary_rows,40,40,True


,check,pass
0,all_output_labels_are_canonical,True
1,no_historical_nonmalignant_labels,True
2,no_malformed_malignant_nonmalignant_labels,True
3,notebook25_labels_used_directly,True
4,historical_aliases_used_for_computation,False


In [11]:
# Diagnostic figures
figure_records = []

plot_data = global_summary.sort_values("frac_pass_10pct", ascending=True)
plt.figure(figsize=(9, max(4, 0.45 * len(plot_data))))
plt.barh(plot_data["candidate_axis"], plot_data["frac_pass_10pct"])
plt.xlabel("Fraction of candidate rows with ligand and receptor expression ≥ 10%")
plt.title("Expression support for LIANA candidate axes")
path = savefig(FIGURE_DIR / f"{OUT_PREFIX}_expression_support_candidate_axes_barplot.png")
figure_records.append({"figure": path.name, "path": str(path), "description": "Global expression support by candidate axis."})

plot_data = overlap_summary.sort_values("n_squidpy_p_le_0_05", ascending=True)
plt.figure(figsize=(9, max(4, 0.45 * len(plot_data))))
plt.barh(plot_data["candidate_axis"], plot_data["n_squidpy_p_le_0_05"])
plt.xlabel("Candidate source-target rows with Squidpy p ≤ 0.05")
plt.title("Squidpy support for LIANA candidate axes")
path = savefig(FIGURE_DIR / f"{OUT_PREFIX}_squidpy_candidate_axes_barplot.png")
figure_records.append({"figure": path.name, "path": str(path), "description": "Significant Squidpy source-target rows by candidate axis."})

figure_manifest = pd.DataFrame(figure_records)
write_tsv_replace(figure_manifest, METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv")
display(figure_manifest)


[FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\SQUIDPY_LR_validation_HVG8000_r0_6\SQUIDPY_LR_validation_HVG8000_r0_6_expression_support_candidate_axes_barplot.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\SQUIDPY_LR_validation_HVG8000_r0_6\SQUIDPY_LR_validation_HVG8000_r0_6_squidpy_candidate_axes_barplot.png
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_figure_manifest.tsv (2, 3)


,figure,path,description
0,SQUIDPY_LR_validation_HVG8000_r0_6_expression_...,G:\Repository\glioma-cnv-single-cell-atlas\fig...,Global expression support by candidate axis.
1,SQUIDPY_LR_validation_HVG8000_r0_6_squidpy_can...,G:\Repository\glioma-cnv-single-cell-atlas\fig...,Significant Squidpy source-target rows by cand...


In [12]:
# Final run summary and output manifest
run_summary = pd.DataFrame([
    {"item": "historical_notebook", "value": "27_squidpy_LR_validation_top_LIANA_axes_HVG8000_r0_6.ipynb"},
    {"item": "canonical_cell_metadata_source", "value": MAIN_CELLS_TSV.name},
    {"item": "canonical_candidate_source", "value": LIANA_CANDIDATE_DETAILS.name},
    {"item": "full_anndata_loaded", "value": False},
    {"item": "memory_safe_selected_gene_streaming", "value": True},
    {"item": "historical_scanpy_preprocessing_reproduced", "value": True},
    {"item": "historical_structural_checkpoints_all_match", "value": bool(historical_audit["match"].all())},
    {"item": "canonical_label_audit_pass", "value": bool(label_audit.loc[label_audit["check"] != "historical_aliases_used_for_computation", "pass"].all())},
    {"item": "squidpy_version", "value": sq.__version__},
    {"item": "n_perms", "value": SQUIDPY_N_PERMS},
    {"item": "threshold", "value": SQUIDPY_THRESHOLD},
    {"item": "random_seed", "value": RANDOM_SEED},
    {"item": "sample_selection", "value": "audited_historical_stable_id_manifest"},
    {"item": "candidate_axes", "value": len(AXES_OF_INTEREST)},
    {"item": "candidate_interactions", "value": len(candidates)},
    {"item": "completed_at", "value": datetime.now().isoformat(timespec="seconds")},
])
write_tsv_replace(run_summary, METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv")

expected_outputs = [
    METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv",
    HISTORICAL_SAMPLE_MANIFEST,
    METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_used.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_memory_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_axis_inventory.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_gene_expression_support_by_celltype_condition.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_expression_support_by_condition.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_expression_support_global.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_expression_support_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_squidpy_run_status.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_squidpy_means_long.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_squidpy_pvalues_long.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_squidpy_candidate_overlap_details.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_squidpy_candidate_overlap_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_thesis_validation_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_historical_checkpoint_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_canonical_label_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv",
]
manifest = pd.DataFrame([
    {
        "file": path.name,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024**2, 3) if path.exists() else np.nan,
    }
    for path in expected_outputs
])
write_tsv_replace(manifest, METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv")
if not manifest["exists"].all():
    raise AssertionError("One or more expected Notebook 30 outputs are missing.")

display(run_summary)
display(manifest)
del adata_sq, X_selected
gc.collect()
print("[DONE] Notebook 30 completed successfully.")


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_run_summary.tsv (16, 2)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\SQUIDPY_LR_validation_HVG8000_r0_6_output_manifest.tsv (20, 4)


,item,value
0,historical_notebook,27_squidpy_LR_validation_top_LIANA_axes_HVG800...
1,canonical_cell_metadata_source,LIANA_prepare_HVG8000_r0_6_main_cells.tsv
2,canonical_candidate_source,LIANA_final_prioritization_HVG8000_r0_6_candid...
3,full_anndata_loaded,False
4,memory_safe_selected_gene_streaming,True
5,historical_scanpy_preprocessing_reproduced,True
6,historical_structural_checkpoints_all_match,True
7,canonical_label_audit_pass,True
8,squidpy_version,1.8.1
9,n_perms,100


,file,path,exists,size_mb
0,SQUIDPY_LR_validation_HVG8000_r0_6_readiness.tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
1,SQUIDPY_LR_validation_HVG8000_r0_6_historical_...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,5.885
2,SQUIDPY_LR_validation_HVG8000_r0_6_celltype_co...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
3,SQUIDPY_LR_validation_HVG8000_r0_6_memory_audi...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
4,SQUIDPY_LR_validation_HVG8000_r0_6_candidate_i...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.190
5,SQUIDPY_LR_validation_HVG8000_r0_6_candidate_a...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
6,SQUIDPY_LR_validation_HVG8000_r0_6_gene_expres...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.066
7,SQUIDPY_LR_validation_HVG8000_r0_6_candidate_e...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.510
8,SQUIDPY_LR_validation_HVG8000_r0_6_candidate_e...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.104
9,SQUIDPY_LR_validation_HVG8000_r0_6_candidate_e...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.004


[DONE] Notebook 30 completed successfully.
